# Chargement des modules

In [130]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Etape 1 - Chargement des fichiers dans le Notebook

In [131]:
def is_csv(filepath: str):
    ext = os.path.splitext(filepath)[1].lower()
    return ext == ".csv"

cwd = os.path.join(os.getcwd(), "data") # cwd est un objet DirEntry, on peut accéder au path via l'attribut

list_dfs = []  # stockera les dfs chargés

for element in os.scandir(cwd):
    if element.is_file() and is_csv(element.path):
        df = pd.read_csv(element.path)
        list_dfs.append(df)

display(len(list_dfs))

5

Explication : Je n'avais pas envie d'écrire 5 read_csv("csv_file_path").

Les DataFrames sont chargés et mis dans une liste. La longueur de la liste est de 5, j'ai donc bien tous les fichiers demandés.


# Etape 2 - A la recherche d'information

Objectifs : 

    - Comprendre l'organisation de chaque DataFrame (head, info, shape)
    
    - Identifier les liens éventuels entre chaque (colonne identique à travers les DF)
    
    - Tirer le maximum d'information (unique, value_counts, describe?, duplicated) 
    

## Premier dataset

In [132]:
first_df = list_dfs[0]

print(f"Le premier fichier comporte {first_df.shape[0]} lignes et {first_df.shape[1]} colonnes")
display(first_df.head(3))

Le premier fichier comporte 241 lignes et 32 colonnes


,Country Code,Short Name,Table Name,Long Name,2-alpha code,Currency Unit,Special Notes,Region,Income Group,WB-2 code,...,IMF data dissemination standard,Latest population census,Latest household survey,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Latest water withdrawal data,Unnamed: 31
0,ABW,Aruba,Aruba,Aruba,AW,Aruban florin,SNA data for 2000-2011 are updated from offici...,Latin America & Caribbean,High income: nonOECD,AW,...,NaN,2010,NaN,NaN,Yes,NaN,NaN,2012.0,NaN,NaN
1,AFG,Afghanistan,Afghanistan,Islamic State of Afghanistan,AF,Afghan afghani,Fiscal year end: March 20; reporting period fo...,South Asia,Low income,AF,...,General Data Dissemination System (GDDS),1979,"Multiple Indicator Cluster Survey (MICS), 2010/11","Integrated household survey (IHS), 2008",NaN,2013/14,NaN,2012.0,2000,NaN
2,AGO,Angola,Angola,People's Republic of Angola,AO,Angolan kwanza,"April 2013 database update: Based on IMF data,...",Sub-Saharan Africa,Upper middle income,AO,...,General Data Dissemination System (GDDS),1970,"Malaria Indicator Survey (MIS), 2011","Integrated household survey (IHS), 2008",NaN,2015,NaN,NaN,2005,NaN


In [133]:
first_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 241 entries, 0 to 240
Data columns (total 32 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   Country Code                                       241 non-null    object 
 1   Short Name                                         241 non-null    object 
 2   Table Name                                         241 non-null    object 
 3   Long Name                                          241 non-null    object 
 4   2-alpha code                                       238 non-null    object 
 5   Currency Unit                                      215 non-null    object 
 6   Special Notes                                      145 non-null    object 
 7   Region                                             214 non-null    object 
 8   Income Group                                       214 non-null    object 
 9   WB-2 code 

In [134]:
first_df.describe()

,National accounts reference year,Latest industrial data,Latest trade data,Unnamed: 31
count,32.00000,107.000000,185.000000,0.0
mean,2001.53125,2008.102804,2010.994595,NaN
std,5.24856,2.616834,2.569675,NaN
min,1987.00000,2000.000000,1995.000000,NaN
25%,1996.75000,2007.500000,2011.000000,NaN
50%,2002.00000,2009.000000,2012.000000,NaN
75%,2005.00000,2010.000000,2012.000000,NaN
max,2012.00000,2010.000000,2012.000000,NaN


En première analyse, nous identifions 241 lignes faisant référence à des pays. Les attributs de ces pays sont au nombre de 32, avec 28 de type littéral (object -> str), et 4 de type numérique (float64). 

Néanmoins, plusieurs observations sont possibles à partir des méthodes .head() et .infos() : 

- Les attributs de format numérique semblent correspondre à des années où certaines actions étatiques ont été effectuées pour la dernière fois. C'est le cas pour les attributs Latest trade data, Latest industrial data, National accounts reference year.
- La dernière colonne (Unnamed: 31), bien qu'au format numérique, est vide (0 non-null valeurs).
- Le nombre de données diffère grandement entre pays, avec par exemple 238 entrées pour l'attribut 2-alpha code, et 32 pour le National accounts reference year.

De ces informations, je peux à présent regarder le nombre de doublons, s'ils existent, supprimer la colonne inutilisable, et calculer la proportion de valeurs manquantes par colonnes.


In [135]:
print(f"Le nombre d'entrée unique est de {(~first_df.duplicated()).values.sum()}, sur {first_df.shape[0]} entrées au total.")

print(first_df["Country Code"][first_df["Country Code"].duplicated() == True])



Le nombre d'entrée unique est de 241, sur 241 entrées au total.
Series([], Name: Country Code, dtype: object)


Avec la première information, nous pouvons voir que toutes les entrées du premier attribut sont uniques. Etant donné que cet attribut correspond à la désignation d'un pays, chaque entrée correspond bien à un pays distinct.

**Note :** Le fait que la Series soit vide dans le deuxième print me confirme qu'aucun doublon n'existe. Bien que la première information soit à mes yeux suffisante, je voulais voir différents moyens de l'obtenir.

Supprimons à présent la colonne inutilisable

In [136]:
first_df.drop(columns={"Unnamed: 31"}, inplace=True)
print(f" Le nombre d'attribut du premier DataFrame est désormais de {first_df.shape[-1]}")

 Le nombre d'attribut du premier DataFrame est désormais de 31


Calculons maintenant la proportion de valeurs manquantes dans chaque attribut

In [137]:
nbr_values = np.array(first_df.count())
columns = list(first_df.columns)

proportion_missing = np.round((1 - (nbr_values / np.max(nbr_values)))*100,decimals = 1)
proportion_missing


prop_missing_per_column = zip(columns, proportion_missing)
prop_missing_per_column = pd.DataFrame(prop_missing_per_column, columns = ["attributes", "missing_values"]).set_index("attributes")
display(prop_missing_per_column)

,missing_values
attributes,
Country Code,0.0
Short Name,0.0
Table Name,0.0
Long Name,0.0
2-alpha code,1.2
Currency Unit,10.8
Special Notes,39.8
Region,11.2
Income Group,11.2


**NOTE :** Pour aller plus loin dans la gestion des colonnes inutilisables, nous pouvons appliquer par exemple une condition. 
Ex : si une colonne a plus de n% de valeurs manquantes, alors on la supprime.
La case ci-dessous réalise cette tâche avec n = 75.

In [138]:
mask = prop_missing_per_column >= 75
columns_to_delete = prop_missing_per_column[mask].dropna().index

#first_df.drop(columns = columns_to_delete, inplace=True)
#first_df

**NOTE IMPORTANTE POUR MOI-MÊME :** Pour supprimer des colonnes, il faut passer par un objet "hashable" - qui utilise une table de hash pour stocker les valeurs (dict, set, frozen set). Donc, utiliser une liste ou tuple ne fonctionne pas. *MAIS* les objets Index des DataFrame sont des objets hashables. 

Les statistiques descriptives de chaque colonne de type numérique ayant déjà été montrées plus haut, regardons le nombre d'occurence de chaque valeur des attributs "littéraux".

Pour cela, je vais d'abord sélectionner les attributs littéraux, puis un à un calculer l'occurrence des valeurs via value_counts. Et pour finir, les stocker dans un dictionnaire avec le nom des attributs comme index.
Je pourrais directement le faire sur le dataframe filtré en entier, mais je trouve le résultat peu lisible (cf case juste en dessous)

%% **QUESTION :** l'entièreté du dataFrame est prise en compte lors du df.value_counts() ? Est-ce qu'une occurrence est comptée comme double quand TOUTES les attributs sont identiques sur deux entrées ?    

In [139]:
#first_df.select_dtypes("object").value_counts()

In [140]:
litteral_columns = list(first_df.select_dtypes("object").columns)

val_occurences_first = {}
for col in litteral_columns:
    val_occurences_first[col] = first_df[col].value_counts()

# Ex : 
val_occurences_first["Government Accounting concept"]

Government Accounting concept
Consolidated central government    95
Budgetary central government       66
Name: count, dtype: int64

## Deuxième dataset

In [141]:
second_df = list_dfs[1]

print(f"Le deuxième fichier comporte {second_df.shape[0]} lignes et {second_df.shape[1]} colonnes")
display(second_df.tail(5))

Le deuxième fichier comporte 886930 lignes et 70 colonnes


,Country Name,Country Code,Indicator Name,Indicator Code,1970,1971,1972,1973,1974,1975,...,2060,2065,2070,2075,2080,2085,2090,2095,2100,Unnamed: 69
886925,Zimbabwe,ZWE,"Youth illiterate population, 15-24 years, male...",UIS.LP.AG15T24.M,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
886926,Zimbabwe,ZWE,"Youth literacy rate, population 15-24 years, b...",SE.ADT.1524.LT.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
886927,Zimbabwe,ZWE,"Youth literacy rate, population 15-24 years, f...",SE.ADT.1524.LT.FE.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
886928,Zimbabwe,ZWE,"Youth literacy rate, population 15-24 years, g...",SE.ADT.1524.LT.FM.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
886929,Zimbabwe,ZWE,"Youth literacy rate, population 15-24 years, m...",SE.ADT.1524.LT.MA.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [142]:
second_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 886930 entries, 0 to 886929
Data columns (total 70 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Country Name    886930 non-null  object 
 1   Country Code    886930 non-null  object 
 2   Indicator Name  886930 non-null  object 
 3   Indicator Code  886930 non-null  object 
 4   1970            72288 non-null   float64
 5   1971            35537 non-null   float64
 6   1972            35619 non-null   float64
 7   1973            35545 non-null   float64
 8   1974            35730 non-null   float64
 9   1975            87306 non-null   float64
 10  1976            37483 non-null   float64
 11  1977            37574 non-null   float64
 12  1978            37576 non-null   float64
 13  1979            36809 non-null   float64
 14  1980            89122 non-null   float64
 15  1981            38777 non-null   float64
 16  1982            37511 non-null   float64
 17  1983      

In [143]:
second_df.describe()

,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,...,2060,2065,2070,2075,2080,2085,2090,2095,2100,Unnamed: 69
count,7.228800e+04,3.553700e+04,3.561900e+04,3.554500e+04,3.573000e+04,8.730600e+04,3.748300e+04,3.757400e+04,3.757600e+04,3.680900e+04,...,5.143600e+04,5.143600e+04,5.143600e+04,5.143600e+04,5.143600e+04,5.143600e+04,5.143600e+04,5.143600e+04,5.143600e+04,0.0
mean,1.974772e+09,4.253638e+09,4.592365e+09,5.105006e+09,5.401493e+09,2.314288e+09,5.731808e+09,6.124437e+09,6.671489e+09,7.436724e+09,...,7.224868e+02,7.271290e+02,7.283779e+02,7.266484e+02,7.228327e+02,7.176899e+02,7.113072e+02,7.034274e+02,6.940296e+02,NaN
std,1.211687e+11,1.804814e+11,1.914083e+11,2.059170e+11,2.112150e+11,1.375059e+11,2.215546e+11,2.325489e+11,2.473986e+11,2.660957e+11,...,2.215845e+04,2.287990e+04,2.352338e+04,2.408149e+04,2.455897e+04,2.496587e+04,2.530183e+04,2.556069e+04,2.574189e+04,NaN
min,-1.435564e+00,-1.594625e+00,-3.056522e+00,-4.032582e+00,-4.213563e+00,-3.658569e+00,-2.950945e+00,-3.174870e+00,-3.558749e+00,-2.973612e+00,...,-1.630000e+00,-1.440000e+00,-1.260000e+00,-1.090000e+00,-9.200000e-01,-7.800000e-01,-6.500000e-01,-5.500000e-01,-4.500000e-01,NaN
25%,8.900000e-01,8.853210e+00,9.240920e+00,9.595200e+00,9.861595e+00,1.400000e+00,9.312615e+00,9.519913e+00,1.000000e+01,1.000000e+01,...,3.000000e-02,3.000000e-02,2.000000e-02,2.000000e-02,1.000000e-02,1.000000e-02,1.000000e-02,1.000000e-02,1.000000e-02,NaN
50%,6.317724e+00,6.316240e+01,6.655139e+01,6.969595e+01,7.087760e+01,9.677420e+00,7.101590e+01,7.133326e+01,7.290512e+01,7.510173e+01,...,2.300000e-01,2.300000e-01,2.300000e-01,2.300000e-01,2.300000e-01,2.300000e-01,2.300000e-01,2.300000e-01,2.200000e-01,NaN
75%,6.251250e+01,5.655200e+04,5.863650e+04,6.202900e+04,6.383675e+04,7.854163e+01,5.682800e+04,5.739175e+04,5.940425e+04,6.411500e+04,...,7.505000e+00,7.500000e+00,7.300000e+00,7.100000e+00,6.722500e+00,6.080000e+00,5.462500e+00,4.680000e+00,4.032500e+00,NaN
max,1.903929e+13,1.986457e+13,2.100916e+13,2.238367e+13,2.282991e+13,2.300634e+13,2.424128e+13,2.521383e+13,2.622101e+13,2.730873e+13,...,2.951569e+06,3.070879e+06,3.169711e+06,3.246239e+06,3.301586e+06,3.337871e+06,3.354746e+06,3.351887e+06,3.330484e+06,NaN


A première vue, le DataFrame représente l'évolution de différents indicateurs à travers les ans (1970 -> 2100). 
On peut alors supposer qu'une partie des attributs est une donnée mesurée, et l'autre est projetée.

Concernant le type des attributs, seuls 4 sont littéraux, le reste étant numériques. Cela fait sens, les 2 premiers sont le code et le nom de pays, les deux suivants sont les indicateurs et leurs codes.

Comme tous les indicateurs sont mélangés, il est difficile de faire des observations quant aux statistiques descriptives sur le dataset initial. On peut néanmoins regarder si d'éventuels doublons sont disséminés parmi les données.

In [144]:
print(f"Le nombre de doublons dans le data est de {second_df[second_df.duplicated()].shape[0]}")

Le nombre de doublons dans le data est de 0


A priori, aucun doublon. On peut déjà supprimer la dernière colonne (Unnamed: 69), qui ne comprend aucune donnée. Suite à cela, on calculera la proportion de données manquantes par attribut

In [145]:
second_df.drop(columns = "Unnamed: 69", inplace= True)

nbr_values = np.array(second_df.count())
columns = list(second_df.columns)

proportion_missing = np.round((1 - (nbr_values / np.max(nbr_values)))*100,decimals = 1)
proportion_missing


prop_missing_per_column = zip(columns, proportion_missing)
prop_missing_per_column = pd.DataFrame(prop_missing_per_column, columns = ["attributes", "missing_values"]).set_index("attributes")
display(prop_missing_per_column)

,missing_values
attributes,
Country Name,0.0
Country Code,0.0
Indicator Name,0.0
Indicator Code,0.0
1970,91.8
...,...
2080,94.2
2085,94.2
2090,94.2


In [146]:
mask = prop_missing_per_column >= 75
year_large_missing_prop = prop_missing_per_column[mask].dropna().index
year_large_missing_prop

Index(['1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978',
       '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987',
       '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996',
       '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005',
       '2006', '2007', '2008', '2009', '2011', '2012', '2013', '2014', '2015',
       '2016', '2017', '2020', '2025', '2030', '2035', '2040', '2045', '2050',
       '2055', '2060', '2065', '2070', '2075', '2080', '2085', '2090', '2095',
       '2100'],
      dtype='object', name='attributes')

Cela ne nous avance pas beaucoup. Si l'on suit le raisonnement évoqué avec le premier dataset, la quasi-totalité des attributs numériques sont à supprimer par manque d'information. Tout supprimer enlèverait beaucoup d'informations et potentiellement de l'information utile.

Un autre raisonnement serait de vérifier non pas les attributs ayant beaucoup de valeurs manquantes, mais plutôt les entrées.
En effet, si certains indicateurs n'ont que très peu d'entrées concernées, alors ceux-là sont réellement fautifs du manque d'information.

Pour cela, on peut grouper le DataFrame par indicateur.

In [147]:
num_attributes = list(second_df.select_dtypes("float64").columns)
test=second_df.groupby(["Indicator Name"])[num_attributes].count()
test



,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,...,2055,2060,2065,2070,2075,2080,2085,2090,2095,2100
Indicator Name,,,,,,,,,,,,,,,,,,,,,
"Adjusted net enrolment rate, lower secondary, both sexes (%)",4,29,29,25,29,22,27,31,31,32,...,0,0,0,0,0,0,0,0,0,0
"Adjusted net enrolment rate, lower secondary, female (%)",2,24,25,22,24,20,22,27,26,26,...,0,0,0,0,0,0,0,0,0,0
"Adjusted net enrolment rate, lower secondary, gender parity index (GPI)",2,24,24,22,24,20,22,27,26,26,...,0,0,0,0,0,0,0,0,0,0
"Adjusted net enrolment rate, lower secondary, male (%)",2,24,24,22,24,20,22,27,26,26,...,0,0,0,0,0,0,0,0,0,0
"Adjusted net enrolment rate, primary, both sexes (%)",15,15,17,18,19,21,22,22,22,22,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Youth illiterate population, 15-24 years, male (number)",3,0,0,0,0,5,8,1,2,7,...,0,0,0,0,0,0,0,0,0,0
"Youth literacy rate, population 15-24 years, both sexes (%)",3,0,0,0,0,5,8,1,2,7,...,0,0,0,0,0,0,0,0,0,0
"Youth literacy rate, population 15-24 years, female (%)",3,0,0,0,0,5,8,1,2,7,...,0,0,0,0,0,0,0,0,0,0


In [148]:
###### A COMPLETER

nbr_values = np.array(test.replace(0,np.nan).count())
#nbr_max_values = len(list(second_df.columns))

proportion_missing = np.round((1 - (nbr_values / np.max(nbr_values)))*100, decimals = 1)
proportion_missing


#prop_missing_per_indicator = zip(indicator, proportion_missing)
#prop_missing_per_column = pd.DataFrame(prop_missing_per_column, columns = ["attributes", "missing_values"]).set_index("attributes")
#display(prop_missing_per_column)

nbr_values

array([ 831,  441,  441,  440,  441,  835,  479,  474,  471,  473,  849,
        496,  481,  473,  483,  858,  505,  503,  498,  503, 1160,  801,
        808,  805,  801, 1325,  838,  862, 1168, 1344, 1928, 1498, 1298,
       1662, 1492, 1969, 1863, 1809, 1594, 1754, 2568, 1939, 2031, 1862,
       1972, 2057,  419,   40,  308,  308,  308,  308,  308,  308,  308,
        308,  308,  308,  308,  308,  308,  308,  308,  308,  308])

Suite à cela, nous pouvons calculer le nombre d'occurrences de chaque colonne catégorielle selon :

In [149]:
litteral_columns = list(second_df.select_dtypes("object").columns)

val_occurences_second = {}
for col in litteral_columns:
    val_occurences_second[col] = second_df[col].value_counts()

# et y accéder par :
val_occurences_second["Country Name"]

Country Name
Arab World                                     3665
East Asia & Pacific                            3665
East Asia & Pacific (excluding high income)    3665
Euro area                                      3665
Europe & Central Asia                          3665
                                               ... 
Virgin Islands (U.S.)                          3665
West Bank and Gaza                             3665
Yemen, Rep.                                    3665
Zambia                                         3665
Zimbabwe                                       3665
Name: count, Length: 242, dtype: int64

## Troisième dataset

In [150]:
third_df = list_dfs[2]

print(f"Le troisième fichier comporte {third_df.shape[0]} lignes et {third_df.shape[1]} colonnes")
display(third_df.head(3))

Le troisième fichier comporte 3665 lignes et 21 colonnes


,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
0,BAR.NOED.1519.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 15-19 with...,Percentage of female population age 15-19 with...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BAR.NOED.1519.ZS,Attainment,Barro-Lee: Percentage of population age 15-19 ...,Percentage of population age 15-19 with no edu...,Percentage of population age 15-19 with no edu...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BAR.NOED.15UP.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 15+ with n...,Percentage of female population age 15+ with n...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [151]:
third_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3665 entries, 0 to 3664
Data columns (total 21 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Series Code                          3665 non-null   object 
 1   Topic                                3665 non-null   object 
 2   Indicator Name                       3665 non-null   object 
 3   Short definition                     2156 non-null   object 
 4   Long definition                      3665 non-null   object 
 5   Unit of measure                      0 non-null      float64
 6   Periodicity                          99 non-null     object 
 7   Base Period                          314 non-null    object 
 8   Other notes                          552 non-null    object 
 9   Aggregation method                   47 non-null     object 
 10  Limitations and exceptions           14 non-null     object 
 11  Notes from original source    

In [152]:
third_df.describe()

,Unit of measure,Notes from original source,Other web links,Related indicators,License Type,Unnamed: 20
count,0.0,0.0,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN


Selon df.info(), nous avons 6 attributs numériques et 15 attributs catégoriels. De plus, les attributs numériques sont exempts de données, étant remplis de NaN. Ce faisant, nous pouvons les supprimer.

In [153]:
num_col = list(third_df.select_dtypes("float64"))
third_df.drop(columns=num_col,inplace=True)
third_df

,Series Code,Topic,Indicator Name,Short definition,Long definition,Periodicity,Base Period,Other notes,Aggregation method,Limitations and exceptions,General comments,Source,Statistical concept and methodology,Development relevance,Related source links
0,BAR.NOED.1519.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 15-19 with...,Percentage of female population age 15-19 with...,NaN,NaN,NaN,NaN,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN
1,BAR.NOED.1519.ZS,Attainment,Barro-Lee: Percentage of population age 15-19 ...,Percentage of population age 15-19 with no edu...,Percentage of population age 15-19 with no edu...,NaN,NaN,NaN,NaN,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN
2,BAR.NOED.15UP.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 15+ with n...,Percentage of female population age 15+ with n...,NaN,NaN,NaN,NaN,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN
3,BAR.NOED.15UP.ZS,Attainment,Barro-Lee: Percentage of population age 15+ wi...,Percentage of population age 15+ with no educa...,Percentage of population age 15+ with no educa...,NaN,NaN,NaN,NaN,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN
4,BAR.NOED.2024.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 20-24 with...,Percentage of female population age 20-24 with...,NaN,NaN,NaN,NaN,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3660,UIS.XUNIT.USCONST.3.FSGOV,Expenditures,Government expenditure per upper secondary stu...,NaN,"Average total (current, capital and transfers)...",NaN,NaN,NaN,NaN,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN
3661,UIS.XUNIT.USCONST.4.FSGOV,Expenditures,Government expenditure per post-secondary non-...,NaN,"Average total (current, capital and transfers)...",NaN,NaN,NaN,NaN,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN
3662,UIS.XUNIT.USCONST.56.FSGOV,Expenditures,Government expenditure per tertiary student (c...,NaN,"Average total (current, capital and transfers)...",NaN,NaN,NaN,NaN,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN
3663,XGDP.23.FSGOV.FDINSTADM.FFD,Expenditures,Government expenditure in secondary institutio...,"Total general (local, regional and central) go...","Total general (local, regional and central) go...",NaN,NaN,Secondary,NaN,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN


La proportion de données manquantes par colonne est de : 

In [154]:
nbr_values = np.array(third_df.count())
columns = list(third_df.columns)

proportion_missing = np.round((1 - (nbr_values / np.max(nbr_values)))*100,decimals = 1)
proportion_missing


prop_missing_per_column = zip(columns, proportion_missing)
prop_missing_per_column = pd.DataFrame(prop_missing_per_column, columns = ["attributes", "missing_values"]).set_index("attributes")
display(prop_missing_per_column)

,missing_values
attributes,
Series Code,0.0
Topic,0.0
Indicator Name,0.0
Short definition,41.2
Long definition,0.0
Periodicity,97.3
Base Period,91.4
Other notes,84.9
Aggregation method,98.7


Étant donné que la majorité des attributs ayant des données manquantes ont un taux > 80 %, je fais le choix de supprimer celles remplissant la condition. J'y ajoute également l'attribut "Short definition", car il est redondant avec "Long definition"

In [155]:
mask = prop_missing_per_column >= 20
columns_to_delete = prop_missing_per_column[mask].dropna().index

third_df.drop(columns = columns_to_delete, inplace=True)
third_df

,Series Code,Topic,Indicator Name,Long definition,Source
0,BAR.NOED.1519.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 15-19 with...,Robert J. Barro and Jong-Wha Lee: http://www.b...
1,BAR.NOED.1519.ZS,Attainment,Barro-Lee: Percentage of population age 15-19 ...,Percentage of population age 15-19 with no edu...,Robert J. Barro and Jong-Wha Lee: http://www.b...
2,BAR.NOED.15UP.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 15+ with n...,Robert J. Barro and Jong-Wha Lee: http://www.b...
3,BAR.NOED.15UP.ZS,Attainment,Barro-Lee: Percentage of population age 15+ wi...,Percentage of population age 15+ with no educa...,Robert J. Barro and Jong-Wha Lee: http://www.b...
4,BAR.NOED.2024.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 20-24 with...,Robert J. Barro and Jong-Wha Lee: http://www.b...
...,...,...,...,...,...
3660,UIS.XUNIT.USCONST.3.FSGOV,Expenditures,Government expenditure per upper secondary stu...,"Average total (current, capital and transfers)...",UNESCO Institute for Statistics
3661,UIS.XUNIT.USCONST.4.FSGOV,Expenditures,Government expenditure per post-secondary non-...,"Average total (current, capital and transfers)...",UNESCO Institute for Statistics
3662,UIS.XUNIT.USCONST.56.FSGOV,Expenditures,Government expenditure per tertiary student (c...,"Average total (current, capital and transfers)...",UNESCO Institute for Statistics
3663,XGDP.23.FSGOV.FDINSTADM.FFD,Expenditures,Government expenditure in secondary institutio...,"Total general (local, regional and central) go...",UNESCO Institute for Statistics


Enfin, je calcule le nombre d'occurrences par colonne catégorielle : 

In [156]:
litteral_columns = list(third_df.select_dtypes("object").columns)

val_occurences_third = {}
for col in litteral_columns:
    val_occurences_third[col] = third_df[col].value_counts()

# et y accéder par :
val_occurences_third["Topic"]

Topic
Learning Outcomes                                                                               1046
Attainment                                                                                       733
Education Equality                                                                               426
Secondary                                                                                        256
Primary                                                                                          248
Population                                                                                       213
Tertiary                                                                                         158
Teachers                                                                                         137
Expenditures                                                                                      93
Engaging the Private Sector (SABER)                                                  

## Quatrième dataset

In [157]:
fourth_df = list_dfs[3]

print(f"Le quatrième fichier comporte {fourth_df.shape[0]} lignes et {fourth_df.shape[1]} colonnes")
display(fourth_df.head(3))

Le quatrième fichier comporte 613 lignes et 4 colonnes


,CountryCode,SeriesCode,DESCRIPTION,Unnamed: 3
0,ABW,SP.POP.TOTL,Data sources : United Nations World Population...,NaN
1,ABW,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
2,AFG,SP.POP.GROW,Data sources: United Nations World Population ...,NaN


In [158]:
fourth_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613 entries, 0 to 612
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CountryCode  613 non-null    object 
 1   SeriesCode   613 non-null    object 
 2   DESCRIPTION  613 non-null    object 
 3   Unnamed: 3   0 non-null      float64
dtypes: float64(1), object(3)
memory usage: 19.3+ KB


In [159]:
fourth_df.describe()

,Unnamed: 3
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


Le quatrième dataset possède 3 attributs catégoriels et un numérique. Le seul attribut numérique est inutilisable, il est donc à supprimer.
Avec ceci, on peut regarder le nombre de doublons éventuels

In [160]:
fourth_df.drop(columns="Unnamed: 3", inplace=True)

In [161]:
print(f"Le nombre de valeur unique est de {(~fourth_df.duplicated()).values.sum()} sur {fourth_df.shape[0]} entrées")

Le nombre de valeur unique est de 613 sur 613 entrées


Finalement, le nombre d'occurrences de chaque valeur dans chaque attribut catégoriciel : 

In [162]:
litteral_columns = list(fourth_df.select_dtypes("object").columns)

val_occurences_fourth = {}
for col in litteral_columns:
    val_occurences_fourth[col] = fourth_df[col].value_counts()

# Ex : 
val_occurences_fourth["SeriesCode"]

SeriesCode
SP.POP.TOTL          211
SP.POP.GROW          211
NY.GDP.PCAP.PP.CD     19
NY.GNP.PCAP.PP.CD     19
NY.GDP.PCAP.PP.KD     19
NY.GNP.MKTP.PP.CD     14
NY.GDP.MKTP.PP.KD     14
NY.GDP.MKTP.PP.CD     14
SP.POP.1564.TO.ZS     13
SP.POP.TOTL.MA.ZS     13
SP.POP.TOTL.FE.ZS     13
SP.POP.0014.TO.ZS     13
NY.GNP.PCAP.CD         6
NY.GDP.PCAP.CD         5
NY.GDP.PCAP.KD         5
SP.POP.1564.MA.IN      4
SP.POP.0014.TO         4
SP.POP.1564.TO         4
SP.POP.1564.FE.IN      4
SP.POP.0014.MA.IN      4
SP.POP.0014.FE.IN      4
Name: count, dtype: int64

## Cinquième dataset

In [163]:
fifth_df = list_dfs[-1]

print(f"Le cinquième fichier comporte {fifth_df.shape[0]} lignes et {fifth_df.shape[1]} colonnes")
display(fifth_df.head(3))

Le cinquième fichier comporte 643638 lignes et 5 colonnes


,CountryCode,SeriesCode,Year,DESCRIPTION,Unnamed: 4
0,ABW,SE.PRE.ENRL.FE,YR2001,Country estimation.,NaN
1,ABW,SE.TER.TCHR.FE,YR2005,Country estimation.,NaN
2,ABW,SE.PRE.TCHR.FE,YR2000,Country estimation.,NaN


In [164]:
fifth_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 643638 entries, 0 to 643637
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   CountryCode  643638 non-null  object 
 1   SeriesCode   643638 non-null  object 
 2   Year         643638 non-null  object 
 3   DESCRIPTION  643638 non-null  object 
 4   Unnamed: 4   0 non-null       float64
dtypes: float64(1), object(4)
memory usage: 24.6+ MB


In [165]:
fifth_df.describe()

,Unnamed: 4
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


Le cinquième dataset est assez semblable au quatrième. Un seul attribut numérique inutilisable, et 3 catégoriels. On reprend donc les mêmes opérations ci-dessous

In [166]:
fifth_df.drop(columns="Unnamed: 4", inplace=True)
print(f"Le nombre de valeur unique est de {(~fourth_df.duplicated()).values.sum()} sur {fourth_df.shape[0]} entrées")

Le nombre de valeur unique est de 613 sur 613 entrées


In [167]:
litteral_columns = list(fifth_df.select_dtypes("object").columns)

val_occurences_fifth = {}
for col in litteral_columns:
    val_occurences_fifth[col] = fifth_df[col].value_counts()

# Ex : 
val_occurences_fifth["SeriesCode"]

SeriesCode
SH.DYN.MORT                  9226
SE.PRM.AGES                  8771
SE.PRM.DURS                  8771
SE.SEC.DURS                  8619
SE.SEC.AGES                  8581
                             ... 
NY.GNP.MKTP.PP.CD               1
NY.GNP.PCAP.PP.CD               1
UIS.AFR.SCHCENRESPR.23.PU       1
SL.UEM.TOTL.MA.ZS               1
UIS.XPubP.0                     1
Name: count, Length: 1558, dtype: int64

# Etape 3 - Filtrage des datasets

Objectif : Filtrer les faux pays  

Le dataset EdStatsCountry.csv présente un attribut intéressant ici : le 2-alpha code.
Ce code commence par un numéro, ou Z ou X quand l'entrée n'est pas un pays.
Ex : 	
- ARB	Arab World	1A    
- EAP	East Asia & Pacific (developing only) 4E
- ECS	Europe & Central Asia (all income levels) Z7
- 125	LIC	Low income	Low income	Low income	XM

Cela dit, il manque des données dans l'attribut 2-alpha code, qui pourraient être complétées par l'attribut WB-2 code.
Ces deux attributs sont les mêmes à première vue, mais nous pouvons les comparer pour en être sûr.

In [185]:
code_mismatch = first_df[first_df["2-alpha code"] != first_df["WB-2 code"]]

code_mismatch

,Country Code,Short Name,Table Name,Long Name,2-alpha code,Currency Unit,Special Notes,Region,Income Group,WB-2 code,...,Government Accounting concept,IMF data dissemination standard,Latest population census,Latest household survey,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Latest water withdrawal data
35,CHI,Channel Islands,Channel Islands,Channel Islands,NaN,Pound sterling,NaN,Europe & Central Asia,High income: nonOECD,JG,...,NaN,NaN,Guernsey: 2009; Jersey: 2011.,NaN,NaN,Yes. Vital registration for Guernsey and Jersey.,NaN,NaN,NaN,NaN
40,COD,Dem. Rep. Congo,"Congo, Dem. Rep.",Democratic Republic of the Congo,CD,Congolese franc,"Based on INS (2000-09) and IMF (2010-13) data,...",Sub-Saharan Africa,Low income,ZR,...,Consolidated central government,General Data Dissemination System (GDDS),1984,"Demographic and Health Survey (DHS), 2013","1-2-3 survey (1-2-3), 2004/05",NaN,NaN,NaN,NaN,2005
158,NAM,Namibia,Namibia,Republic of Namibia,NaN,Namibian dollar,Fiscal year end: March 31; reporting period fo...,Sub-Saharan Africa,Upper middle income,NaN,...,Budgetary central government,General Data Dissemination System (GDDS),2011,"Demographic and Health Survey (DHS), 2013","Expenditure survey/budget survey (ES/BS), 2009/10",NaN,2014,NaN,2012.0,2002
181,PSE,West Bank and Gaza,West Bank and Gaza,West Bank and Gaza,PS,Israeli new shekel,NaN,Middle East & North Africa,Lower middle income,GZ,...,Budgetary central government,Special Data Dissemination Standard (SDDS),2007,"Multiple Indicator Cluster Survey (MICS), 2010","Integrated household survey (IHS), 2009",NaN,NaN,2010.0,NaN,2005
197,SRB,Serbia,Serbia,Republic of Serbia,RS,New Serbian dinar,Montenegro declared independence from Serbia a...,Europe & Central Asia,Upper middle income,YF,...,Consolidated central government,General Data Dissemination System (GDDS),2011,"Multiple Indicator Cluster Survey (MICS), 2010","Integrated household survey (IHS), 2010",Yes,2012,2010.0,NaN,2009
216,TLS,Timor-Leste,Timor-Leste,Democratic Republic of Timor-Leste,TL,U.S. dollar,"Based on official government statistics, natio...",East Asia & Pacific,Lower middle income,TP,...,NaN,General Data Dissemination System (GDDS),2010,"Demographic and Health Survey (DHS), 2009/10",Living Standards Measurement Study Survey (LSM...,NaN,2010. Population and Housing Census.,NaN,2005.0,2004
236,XKX,Kosovo,Kosovo,Republic of Kosovo,NaN,Euro,"Kosovo became a World Bank member on June 29, ...",Europe & Central Asia,Lower middle income,KV,...,NaN,General Data Dissemination System (GDDS),2011,NaN,"Integrated household survey (IHS), 2011",NaN,NaN,NaN,NaN,NaN
237,YEM,Yemen,"Yemen, Rep.",Republic of Yemen,YE,Yemeni rial,Based on official government statistics and In...,Middle East & North Africa,Lower middle income,RY,...,Budgetary central government,General Data Dissemination System (GDDS),2004,"Demographic and Health Survey (DHS), 2013","Expenditure survey/budget survey (ES/BS), 2005",NaN,NaN,2006.0,2012.0,2005


Effectivement, nous avons 8 occurrences de non-correspondance entre les deux codes.
Pour ces non-correspondances : 

- 2 n'ont pas d'alpha code, mais ont un WB répondant à nos conditions préalablement établies. (West Bank et Kosovo)
- 1 n'a aucun code (Channel Islands)
- 5 ont deux codes différents, mais l'alpha code répond aux conditions précédemment établies.

Dans tous les cas, les non-occurrences ne concernent que des pays, pas d'autres entités.

On peut donc adopter la stratégie de masquer le dataframe selon l'alpha code, et de ne pas ignorer les NaN.